In [2]:
import subprocess
import sys

def ensure_libraries():
    # Maps the import name to the actual pip installation package name
    required_packages = {
        "easyocr": "easyocr",
        "PyPDF2": "PyPDF2",
        "thefuzz": "thefuzz[speed]", # Includes python-Levenshtein for 4-10x faster string matching
        "PIL": "pillow",              # Imported as PIL, installed as pillow
        "matplotlib": "matplotlib",
        "numpy": "numpy",
        "streamlit": "streamlit"
    }

    print("Checking project dependencies...\n")
    for module_name, pip_name in required_packages.items():
        try:
            # Try to import the module to check existence
            __import__(module_name)
            print(f"✅ '{module_name}' is ready.")
        except ImportError:
            print(f"⚠️ '{module_name}' not found. Installing package '{pip_name}'...")
            try:
                # Install the package into the current active environment
                subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
                print(f"   Successfully installed '{pip_name}'.")
            except subprocess.CalledProcessError as e:
                print(f"❌ Failed to install '{pip_name}': {e}")
                sys.exit(1)

# Step 1: Verify and install all missing dependencies
ensure_libraries()

import io
import easyocr
import PyPDF2
from PIL import Image, ImageDraw, ImageFont
from thefuzz import process, fuzz
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np # Import numpy
import matplotlib.pyplot as plt # Import matplotlib

# 2. Initialize OCR for Chinese Simplified and English
reader = easyocr.Reader(['ch_sim', 'en'])

# 3. Create Interface Elements
upload_image = widgets.FileUpload(accept='image/*', multiple=False, description="Upload Image")
upload_pdf = widgets.FileUpload(accept='.pdf', multiple=False, description="Upload PDF")
si_threshold = widgets.IntSlider(value=70, min=30, max=100, step=5, description='Match Sensitivity %:', style={'description_width': 'initial'})
btn_translate = widgets.Button(description="Overlay Translation", button_style='success')
output_area = widgets.Output()

print("--- 1. UPLOAD FILES & ADJUST SENSITIVITY ---")
display(widgets.HBox([upload_image, upload_pdf]))
display(si_threshold)
print("\n--- 2. EXECUTE OVERLAY ---")
display(btn_translate, output_area)

# 4. Processing Core
def process_and_overlay(b):
    with output_area:
        clear_output()

        if not upload_image.value:
            print("❌ Please upload an image file.")
            return
        if not upload_pdf.value:
            print("❌ Please upload your PDF translation file.")
            return

        print("⏳ Parsing PDF dictionary and scanning image text... Please wait...")

        try:
            # --- PARSE PDF TRANSLATION DICTIONARY ---
            pdf_data = list(upload_pdf.value.values())[0]['content']
            pdf_file = io.BytesIO(pdf_data)
            pdf_reader = PyPDF2.PdfReader(pdf_file)

            translation_map = {}
            excel_choices = []

            # Read text page-by-page from the PDF
            for page in pdf_reader.pages:
                page_text = page.extract_text()
                if not page_text:
                    continue

                # Split text into lines
                lines = page_text.split('\n')
                for line in lines:
                    line = line.strip()
                    if not line:
                        continue

                    # Handle split structures. Assumes standard dictionary formatting like:
                    # '张飞 Zhang Fei' OR '张飞,Zhang Fei' OR '张飞: Zhang Fei'
                    parts = []
                    if ',' in line:
                        parts = line.split(',', 1)
                    elif ':' in line:
                        parts = line.split(':', 1)
                    else:
                        # Fallback: Split on the first whitespace boundary separating Chinese characters from English
                        # Finds where Chinese ends and English text begins
                        for idx, char in enumerate(line):
                            if idx > 0 and ord(char) < 128 and ord(line[idx-1]) >= 128:
                                parts = [line[:idx].strip(), line[idx:].strip()]
                                break

                    if len(parts) == 2:
                        source_text = parts[0].strip()
                        translation_text = parts[1].strip()

                        if source_text and translation_text:
                            translation_map[source_text.lower()] = translation_text
                            excel_choices.append(source_text)

            if not translation_map:
                print("❌ Could not extract any valid Source/Translation pairs from the PDF layout.")
                print("💡 Ensure your PDF format maps terms cleanly (e.g., '张飞, Zhang Fei' or '张飞: Zhang Fei').")
                return

            # --- PROCESS AND OVERLAY TARGET IMAGE ---
            image_data = list(upload_image.value.values())[0]['content']
            original_image = Image.open(io.BytesIO(image_data)).convert("RGBA") # Keep original for side-by-side
            base_image = original_image.copy() # Create a copy to modify

            # Read image layout text blocks (convert PIL Image to NumPy array for EasyOCR)
            ocr_results = reader.readtext(np.array(base_image), detail=1)

            if not ocr_results:
                print("❓ No text blocks detected in the image.")
                return

            draw_layer = ImageDraw.Draw(base_image)
            font = ImageFont.load_default()

            user_threshold = si_threshold.value
            matches_count = 0

            for box, text_raw, confidence in ocr_results:
                text_clean = text_raw.strip()
                if not text_clean:
                    continue

                # Fuzzy matching calculation
                best_match, score = process.extractOne(text_clean, excel_choices, scorer=fuzz.token_sort_ratio)

                if score >= user_threshold:
                    translation_text = str(translation_map[best_match.lower()])
                    matches_count += 1

                    # Compute rectangular corners from point coordinates matrix
                    x_coords = [p[0] for p in box]
                    y_coords = [p[1] for p in box]
                    x_min, y_min = int(min(x_coords)), int(min(y_coords))
                    x_max, y_max = int(max(x_coords)), int(max(y_coords))

                    # Paint dark mask panel block over old Chinese character positions
                    draw_layer.rectangle([x_min, y_min, x_max, y_max], fill=(15, 15, 15, 245))

                    # Write English text inside boundary box
                    draw_layer.text((x_min + 4, y_min + 2), translation_text, fill=(255, 255, 255, 255), font=font)

            print(f"✨ Successfully replaced {matches_count} text blocks!\n")

            # Display original and translated images side-by-side
            fig, axes = plt.subplots(1, 2, figsize=(15, 7))
            axes[0].imshow(original_image.convert("RGB"))
            axes[0].set_title("Original Image")
            axes[0].axis('off')

            axes[1].imshow(base_image.convert("RGB"))
            axes[1].set_title("Translated Overlay")
            axes[1].axis('off')

            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"An unexpected error occurred: {str(e)}")

# Bind logic engine execution to button click
btn_translate.on_click(process_and_overlay)


Checking project dependencies...

✅ 'easyocr' is ready.
✅ 'PyPDF2' is ready.
⚠️ 'thefuzz' not found. Installing package 'thefuzz[speed]'...
✅ 'PIL' is ready.
✅ 'matplotlib' is ready.
✅ 'numpy' is ready.
⚠️ 'streamlit' not found. Installing package 'streamlit'...


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete--- 1. UPLOAD FILES & ADJUST SENSITIVITY ---


IntSlider(value=70, description='Match Sensitivity %:', min=30, step=5, style=SliderStyle(description_width='i…


--- 2. EXECUTE OVERLAY ---


Button(button_style='success', description='Overlay Translation', style=ButtonStyle())

Output()